# 02 — Modeling dan tuning
Gunakan machine learning tradisional saja. Notebook memanggil modul agar kode training dan evaluasi konsisten dengan CLI. Training penuh dapat berlangsung lama.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
assert (ROOT / 'scripts').is_dir(), 'Buka notebook dari root proyek atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
from scripts.train import build_pipeline, run_training, run_restored_cv
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
pipeline = build_pipeline('native_lightgbm', use_id_features=True)
assert isinstance(pipeline, Pipeline)
assert isinstance(pipeline.named_steps['preprocess'], ColumnTransformer)
pipeline


## Konfigurasi eksperimen
Mode native_cv menggunakan 5-fold V2, 500 pohon tetap, dan tuning bobot keputusan pada fold 0. Empat fold lain untuk konfirmasi. Deskripsi holdout berikut berlaku untuk mode baseline.
Holdout 20% tidak dipakai tuning. RandomizedSearchCV memakai 3-fold di bagian training. Skor holdout dicatat sebelum refit seluruh training. Untuk uji cepat, ubah SAMPLE_SIZE menjadi 2000; hasil subset bukan skor kompetisi.

In [ ]:
MODE = 'native_cv'  # rekomendasi; 'baseline' untuk perbandingan one-hot
if MODE == 'native_cv':
    report = run_restored_cv()
else:
    report = run_training(model_name='lightgbm', tune=True,
                          output=ROOT / 'models/model.pkl')
report


## Interpretasi
Periksa F1 tiap kelas pada classification_report, bukan hanya accuracy. Bandingkan eksperimen menggunakan protokol split yang sama. Hasil V2 historis 0.582838 adalah CV pipeline lama, bukan hasil notebook ini. Setiap fold native menyimpan preprocessing dan classifier dalam satu .pkl; native_ensemble.pkl memuat kelima fold. Model baseline terpisah memakai model.pkl.